# Recursion & Backtracking in Python — Principles, Call Stack & Algorithmic Patterns

> **Topic:** Recursion & Backtracking | **Folder:** Data Structures & Algorithms

**Recursion** is a programming technique where a function calls itself to solve smaller instances of a problem.
Every recursive algorithm requires two components:
1. **Base Case**: The condition that terminates recursion and prevents infinite loops.
2. **Recursive Step**: Reduces the problem instance toward the base case.

**Backtracking** is a systematic algorithmic technique for searching decision trees by **exploring candidate solutions**
and **pruning (backing track)** as soon as a candidate fails to satisfy constraints.

---

## Table of Contents
1. [Recursion Fundamentals & The Call Stack](#1.-Recursion-Fundamentals-&-The-Call-Stack)
2. [Types of Recursion (Tail Recursion & TCO)](#2.-Types-of-Recursion-(Tail-Recursion-&-TCO))
3. [Recursion vs. Iteration](#3.-Recursion-vs.-Iteration)
4. [Recursion Limits & Memoization (`sys.getrecursionlimit`, `@lru_cache`)](#4.-Recursion-Limits-&-Memoization-(sys.getrecursionlimit,-@lru_cache))
5. [Algorithmic Pattern 1: Divide & Conquer (Merge Sort & Binary Search)](#5.-Algorithmic-Pattern-1:-Divide-&-Conquer-(Merge-Sort-&-Binary-Search))
6. [Algorithmic Pattern 2: Backtracking — Subsets & Power Set](#6.-Algorithmic-Pattern-2:-Backtracking-—-Subsets-&-Power-Set)
7. [Algorithmic Pattern 3: Backtracking — Permutations](#7.-Algorithmic-Pattern-3:-Backtracking-—-Permutations)
8. [Algorithmic Pattern 4: Backtracking — N-Queens Problem](#8.-Algorithmic-Pattern-4:-Backtracking-—-N-Queens-Problem)
9. [Algorithmic Pattern 5: Nested Structure Recursion (Tree Flattening)](#9.-Algorithmic-Pattern-5:-Nested-Structure-Recursion-(Tree-Flattening))
10. [Quick Reference Card](#10.-Quick-Reference-Card)


---
## 1. Recursion Fundamentals & The Call Stack

When a recursive function is invoked, a **stack frame** storing local variables and return addresses
is pushed onto the system call stack. Stack frames are popped as base cases return.


In [ ]:
# Classic Recursion: Factorial
def factorial(n):
    if n <= 1:           # Base Case
        return 1
    return n * factorial(n - 1)  # Recursive Step

print("5! =", factorial(5))


---
## 2. Types of Recursion (Tail Recursion & TCO)

- **Head / Standard Recursion**: Additional work (multiplication, addition) is performed *after* the recursive call returns.
- **Tail Recursion**: The recursive call is the **very last operation** performed by the function.

> **Note**: Python does **NOT** implement Tail Call Optimization (TCO) for architectural simplicity and clear stack tracebacks.


In [ ]:
# Tail-Recursive Accumulator Pattern
def factorial_tail(n, accumulator=1):
    if n <= 1:
        return accumulator
    return factorial_tail(n - 1, n * accumulator)  # Tail call

print("5! (Tail Recursive) =", factorial_tail(5))


---
## 3. Recursion vs. Iteration

Any recursive function can be rewritten iteratively using an **explicit stack** data structure.


In [ ]:
# Converting Recursive Fibonacci to Iterative
def fib_iterative(n):
    if n <= 1: return n
    a, b = 0, 1
    for _ in range(2, n + 1):
        a, b = b, a + b
    return b

print("Fibonacci(10) Iterative:", fib_iterative(10))


---
## 4. Recursion Limits & Memoization (`@lru_cache`)

Overlapping subproblems cause exponential $O(2^n)$ calls.  
Using `@functools.lru_cache` memoizes results, reducing complexity to **$O(n)$ time**.


In [ ]:
import sys
from functools import lru_cache

print("Current recursion limit:", sys.getrecursionlimit())

@lru_cache(maxsize=None)
def fib_memo(n):
    if n <= 1: return n
    return fib_memo(n - 1) + fib_memo(n - 2)

print("Fibonacci(50) Memoized:", fib_memo(50))


---
## 5. Algorithmic Pattern 1: Divide & Conquer (Merge Sort)

Divides the problem into subproblems, solves subproblems recursively, and combines results in **$O(n \log n)$ time**.


In [ ]:
def merge_sort(arr):
    if len(arr) <= 1: return arr
    mid = len(arr) // 2
    left = merge_sort(arr[:mid])
    right = merge_sort(arr[mid:])
    return merge(left, right)

def merge(left, right):
    res = []
    i = j = 0
    while i < len(left) and j < len(right):
        if left[i] <= right[j]: res.append(left[i]); i += 1
        else: res.append(right[j]); j += 1
    res.extend(left[i:])
    res.extend(right[j:])
    return res

data = [38, 27, 43, 3, 9, 82, 10]
print("Merge Sorted:", merge_sort(data))


---
## 6. Algorithmic Pattern 2: Backtracking — Subsets & Power Set

Generates all $2^n$ subsets of a set by making inclusion/exclusion decisions at each element.


In [ ]:
def subsets(nums):
    res = []
    def backtrack(index, path):
        if index == len(nums):
            res.append(list(path))
            return
        # Choice 1: Exclude nums[index]
        backtrack(index + 1, path)
        # Choice 2: Include nums[index]
        path.append(nums[index])
        backtrack(index + 1, path)
        path.pop()  # Backtrack step

    backtrack(0, [])
    return res

print("Subsets of [1, 2, 3]:", subsets([1, 2, 3]))


---
## 7. Algorithmic Pattern 3: Backtracking — Permutations

Generates all $n!$ permutations by placing unvisited elements in each position.


In [ ]:
def permutations(nums):
    res = []
    visited = [False] * len(nums)
    
    def backtrack(path):
        if len(path) == len(nums):
            res.append(list(path))
            return
        for i in range(len(nums)):
            if not visited[i]:
                visited[i] = True
                path.append(nums[i])
                backtrack(path)
                path.pop()       # Backtrack
                visited[i] = False

    backtrack([])
    return res

print("Permutations of [1, 2, 3]:", permutations([1, 2, 3]))


---
## 8. Algorithmic Pattern 4: Backtracking — N-Queens Problem

Places $N$ non-attacking queens on an $N \times N$ chessboard using column/diagonal constraint tracking.


In [ ]:
def solveNQueens(n):
    cols = set()
    pos_diag = set()  # (r + c)
    neg_diag = set()  # (r - c)
    res = []
    board = [["."] * n for _ in range(n)]

    def backtrack(r):
        if r == n:
            res.append(["".join(row) for row in board])
            return
        for c in range(n):
            if c in cols or (r + c) in pos_diag or (r - c) in neg_diag:
                continue
            cols.add(c); pos_diag.add(r + c); neg_diag.add(r - c)
            board[r][c] = "Q"
            backtrack(r + 1)
            board[r][c] = "."  # Backtrack
            cols.remove(c); pos_diag.remove(r + c); neg_diag.remove(r - c)

    backtrack(0)
    return res

solutions_4_queens = solveNQueens(4)
print(f"4-Queens Total Solutions: {len(solutions_4_queens)}")
print("Sample Solution Board:")
for row in solutions_4_queens[0]: print(" ", row)


---
## 9. Algorithmic Pattern 5: Nested Structure Recursion (Tree Flattening)


In [ ]:
def flatten_nested_list(nested):
    res = []
    for item in nested:
        if isinstance(item, list):
            res.extend(flatten_nested_list(item))  # Recurse
        else:
            res.append(item)
    return res

nested_data = [1, [2, [3, 4], 5], [6, [7]]]
print("Flattened List:", flatten_nested_list(nested_data))


---
## 10. Quick Reference Card


In [ ]:
# ==================================================================
# RECURSION & BACKTRACKING – QUICK REFERENCE
# ==================================================================

# Subsets / Combinations Template:
# def backtrack(start, path):
#     res.append(list(path))
#     for i in range(start, len(nums)):
#         path.append(nums[i])
#         backtrack(i + 1, path)
#         path.pop()  # Backtrack


---
## Summary

| Technique | Complexity | Key Property |
|-----------|------------|--------------|
| **Standard Recursion** | $O(n)$ stack space | Requires explicit base case |
| **Memoized Recursion** | $O(n)$ time | `@lru_cache` eliminates overlapping subproblems |
| **Divide & Conquer** | $O(n \log n)$ | Split $\rightarrow$ Recurse $\rightarrow$ Combine (Merge Sort) |
| **Backtracking (Subsets)** | $O(2^n)$ | Choose $\rightarrow$ Explore $\rightarrow$ Un-choose (prune) |
| **Backtracking (Permutations)** | $O(n!)$ | Decision tree for positional permutations |

---
*Next up: **Sorting & Searching Algorithms***
